# AI Voice Agent with Google Gemini

This notebook implements an AI-powered voice agent that:
1. Takes voice input from microphone OR accepts audio file uploads via web API
2. Transcribes audio to text
3. **Sends the text to Google Gemini AI for intelligent responses**
4. Returns text responses (JSON format) OR audio responses (MP3 format)

## Features

- **AI-Powered**: Uses Google Gemini AI for intelligent responses
- **Web API**: Hosted with ngrok for remote access
- **Audio Processing**: Accepts audio files (WAV, MP3, M4A, FLAC, OGG, WEBM)
- **Text Response**: Returns AI responses as JSON text
- **Audio Response**: Returns AI responses as MP3 audio files (text-to-speech)

## Setup Instructions

1. Install required packages (run the next cell)
2. For local mode: Make sure you have a microphone connected
3. For web API mode: Run the server cells to start the API with ngrok
4. Run the cells in order

In [ ]:
# Install required packages
!apt-get install -y portaudio19-dev
%pip install speechrecognition pyaudio requests beautifulsoup4 flask flask-cors pyngrok pydub google-genai gtts --quiet

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libasound2-dev libjack-dev libjack0 libportaudio2 libportaudiocpp0
Suggested packages:
  libasound2-doc jackd1 portaudio19-doc
The following packages will be REMOVED:
  libjack-jackd2-0
The following NEW packages will be installed:
  libasound2-dev libjack-dev libjack0 libportaudio2 libportaudiocpp0
  portaudio19-dev
0 upgraded, 6 newly installed, 1 to remove and 41 not upgraded.
Need to get 596 kB of archives.
After this operation, 3,178 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libjack0 amd64 1:0.125.0-3build2 [93.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libasound2-dev amd64 1.2.6.1-1ubuntu1 [110 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libjack-dev amd64 1:0.125.0-3build2 [206 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/univers

In [ ]:
# Import required libraries
import speech_recognition as sr
import requests
from bs4 import BeautifulSoup
import re
import os
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from werkzeug.utils import secure_filename
import tempfile
from pyngrok import ngrok
import threading
import time
from gtts import gTTS
import io

# Initialize Google Gemini AI
from google import genai

# Set Gemini API key
GEMINI_API_KEY = "<auth_token_here>"

# Initialize Gemini client (as per user's example)
client = genai.Client(api_key=GEMINI_API_KEY)

print("Google Gemini AI client initialized!")

Google Gemini AI client initialized!


In [ ]:
# def listen():
#     """Listen to microphone input and convert to text"""
#     with microphone as source:
#         print("Listening...")
#         try:
#             # Listen for audio with timeout
#             audio = recognizer.listen(source, timeout=5, phrase_time_limit=10)
#             print("Processing...")

#             # Recognize speech using Google's speech recognition
#             text = recognizer.recognize_google(audio)
#             print(f"You said: {text}")
#             return text
#         except sr.WaitTimeoutError:
#             print("No speech detected. Please try again.")
#             return None
#         except sr.UnknownValueError:
#             print("Could not understand audio. Please try again.")
#             return None
#         except sr.RequestError as e:
#             print(f"Could not request results from speech recognition service; {e}")
#             return None

def process_audio_file(audio_file_path):
    """Process an audio file and convert it to text"""
    try:
        recognizer = sr.Recognizer()

        # Check if file needs conversion (speech_recognition works best with WAV)
        try:
            # Load audio file
            with sr.AudioFile(audio_file_path) as source:
                # Adjust for ambient noise
                recognizer.adjust_for_ambient_noise(source, duration=0.5)
                # Read the audio file
                audio = recognizer.record(source)
        except ValueError:
            # If AudioFile can't read it, try converting with pydub
            try:
                from pydub import AudioSegment
                # Convert to WAV format
                audio_data = AudioSegment.from_file(audio_file_path)
                wav_path = audio_file_path.rsplit('.', 1)[0] + '.wav'
                audio_data.export(wav_path, format="wav")

                with sr.AudioFile(wav_path) as source:
                    recognizer.adjust_for_ambient_noise(source, duration=0.5)
                    audio = recognizer.record(source)

                # Clean up converted file
                if os.path.exists(wav_path) and wav_path != audio_file_path:
                    os.remove(wav_path)
            except Exception as conv_error:
                print(f"Error converting audio format: {conv_error}")
                return None

        # Recognize speech using Google's speech recognition
        text = recognizer.recognize_google(audio)
        print(f"Transcribed text: {text}")
        return text
    except sr.UnknownValueError:
        print("Could not understand audio from file.")
        return None
    except sr.RequestError as e:
        print(f"Could not request results from speech recognition service; {e}")
        return None
    except Exception as e:
        print(f"Error processing audio file: {e}")
        return None

In [ ]:
def search_web(query):
    """Search the web for information about the query"""
    try:
        # Use DuckDuckGo search (no API key needed)
        search_url = f"https://html.duckduckgo.com/html/?q={query.replace(' ', '+')}"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

        response = requests.get(search_url, headers=headers, timeout=5)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Try to extract first result snippet
        results = soup.find_all('a', class_='result__snippet')
        if results:
            # Get first result and clean it
            result_text = results[0].get_text()
            # Clean up the text
            result_text = re.sub(r'\s+', ' ', result_text).strip()
            return result_text[:500]  # Limit to 500 characters

        return None
    except Exception as e:
        print(f"Error in web search: {e}")
        return None

In [ ]:
def get_answer_using_gemini(query):
    """Get answer using Google Gemini AI"""
    try:
        # Modify the query for a 5-year-old kid and limit to 20 words
        modified_query = f"Explain '{query}' to a 5-year-old kid in a maximum of 20 words."

        # Try the model name from user's example
        response = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=modified_query,
        )
        return response.text.strip()
    except Exception as e:
        print(f"Gemini AI error with gemini-3-flash-preview: {e}")
        # Try with alternative model names
        alternative_models = ["gemini-2.0-flash-exp", "gemini-1.5-flash", "gemini-1.5-pro"]
        for model_name in alternative_models:
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=modified_query,
                )
                return response.text.strip()
            except Exception as e2:
                print(f"Gemini AI error with {model_name}: {e2}")
                continue
        return None

In [ ]:
def process_query(query):
    """Process the user's query using Gemini AI and return an answer"""
    if not query:
        return "I didn't catch that. Could you please repeat?"

    query_lower = query.lower().strip()

    # Handle greetings
    if any(word in query_lower for word in ['hello', 'hi', 'hey', 'greetings']):
        return "Hello! I'm your AI voice assistant powered by Google Gemini. How can I help you today?"

    # Handle goodbye
    if any(word in query_lower for word in ['goodbye', 'bye', 'exit', 'quit', 'stop']):
        return "Goodbye! Have a great day!"

    # Use Gemini AI to process the query
    print(f"Sending query to Gemini AI: {query}")
    answer = get_answer_using_gemini(query)

    if answer:
        return answer

    # Fallback response if Gemini fails
    return f"I understand you asked about '{query}'. I'm having trouble processing that right now. Could you try rephrasing your question?"

def text_to_speech(text, lang='en'):
    """Convert text to speech audio file"""
    try:
        # Create gTTS object
        tts = gTTS(text=text, lang=lang, slow=False)

        # Save to a BytesIO buffer
        audio_buffer = io.BytesIO()
        tts.write_to_fp(audio_buffer)
        audio_buffer.seek(0)

        return audio_buffer
    except Exception as e:
        print(f"Error converting text to speech: {e}")
        return None

In [ ]:
# def voice_agent_loop():
#     """Main loop for the voice agent"""
#     speak("Voice agent is ready. How can I help you?")

#     while True:
#         # Listen for user input
#         user_input = listen()

#         if user_input is None:
#             continue

#         # Process the query
#         response = process_query(user_input)

#         # Speak the response
#         speak(response)

#         # Check if user wants to exit
#         if any(word in user_input.lower() for word in ['goodbye', 'bye', 'exit', 'quit', 'stop']):
#             break

#     speak("Voice agent session ended. Thank you!")

## Usage

Run the cell below to start the voice agent.

**Note:**
- Make sure your microphone is connected and working
- Speak clearly and wait for the "Listening..." prompt
- Say "goodbye", "exit", or "quit" to end the session
- For better answers, you can set up an OpenAI API key as an environment variable: `OPENAI_API_KEY`

In [ ]:
# # Start the voice agent (local mode) - Commented out as per user request
# # voice_agent_loop()

## Web API Mode with Ngrok

The following cells set up a Flask web server that:
- Accepts audio file uploads OR text input via POST request
- Transcribes the audio to text (if audio provided)
- Sends the text to Gemini AI for processing
- Returns the AI response as JSON text OR MP3 audio file

In [ ]:
# Initialize Flask app
app = Flask(__name__)
CORS(app)  # Enable CORS for cross-origin requests

# Configure upload settings
UPLOAD_FOLDER = tempfile.gettempdir()
ALLOWED_EXTENSIONS = {'wav', 'mp3', 'm4a', 'flac', 'ogg', 'webm'}

app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16MB max file size

def allowed_file(filename):
    """Check if file extension is allowed"""
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

print("Flask app initialized!")

Flask app initialized!


In [ ]:
@app.route('/health', methods=['GET'])
def health_check():
    """Health check endpoint"""
    return jsonify({
        'status': 'healthy',
        'message': 'AI Voice Agent API is running',
        'ai_model': 'Google Gemini',
        'endpoints': {
            'process_audio': '/process-audio (POST) - Upload audio file, get text response',
            'process_text': '/process-text (POST) - Send text, get text response',
            'process_audio_to_audio': '/process-audio-to-audio (POST) - Upload audio file, get audio response',
            'process_text_to_audio': '/process-text-to-audio (POST) - Send text, get audio response'
        }
    }), 200

@app.route('/process-audio', methods=['POST'])
def process_audio():
    """Endpoint to process uploaded audio file - returns text response"""
    uploaded_filepath = None

    try:
        # Check if file is present
        if 'audio' not in request.files:
            return jsonify({'error': 'No audio file provided'}), 400

        file = request.files['audio']

        if file.filename == '':
            return jsonify({'error': 'No file selected'}), 400

        if not allowed_file(file.filename):
            return jsonify({'error': f'File type not allowed. Allowed types: {ALLOWED_EXTENSIONS}'}), 400

        # Save uploaded file
        filename = secure_filename(file.filename)
        uploaded_filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(uploaded_filepath)

        print(f"Received audio file: {filename}")

        # Process audio file to text
        transcribed_text = process_audio_file(uploaded_filepath)

        if not transcribed_text:
            return jsonify({'error': 'Could not transcribe audio. Please ensure the audio file contains clear speech.'}), 400

        print(f"Transcribed: {transcribed_text}")

        # Process the query using Gemini AI
        response_text = process_query(transcribed_text)

        print(f"AI Response: {response_text}")

        # Return JSON response with text
        return jsonify({
            'transcribed_text': transcribed_text,
            'response': response_text,
            'ai_model': 'Google Gemini'
        }), 200

    except Exception as e:
        print(f"Error processing request: {e}")
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500
    finally:
        # Clean up uploaded file
        if uploaded_filepath and os.path.exists(uploaded_filepath):
            try:
                os.remove(uploaded_filepath)
            except:
                pass

@app.route('/process-text', methods=['POST'])
def process_text():
    """Endpoint to process text directly - returns text response"""
    try:
        data = request.get_json()

        if not data or 'text' not in data:
            return jsonify({'error': 'No text provided'}), 400

        query = data['text']
        print(f"Received text query: {query}")

        # Process the query using Gemini AI
        response_text = process_query(query)

        print(f"AI Response: {response_text}")

        # Return JSON response with text
        return jsonify({
            'query': query,
            'response': response_text,
            'ai_model': 'Google Gemini'
        }), 200

    except Exception as e:
        print(f"Error processing request: {e}")
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

@app.route('/process-audio-to-audio', methods=['POST'])
def process_audio_to_audio():
    """Endpoint to process uploaded audio file - returns audio response"""
    uploaded_filepath = None
    audio_filepath = None

    try:
        # Check if file is present
        if 'audio' not in request.files:
            return jsonify({'error': 'No audio file provided'}), 400

        file = request.files['audio']

        if file.filename == '':
            return jsonify({'error': 'No file selected'}), 400

        if not allowed_file(file.filename):
            return jsonify({'error': f'File type not allowed. Allowed types: {ALLOWED_EXTENSIONS}'}), 400

        # Save uploaded file
        filename = secure_filename(file.filename)
        uploaded_filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(uploaded_filepath)

        print(f"Received audio file: {filename}")

        # Process audio file to text
        transcribed_text = process_audio_file(uploaded_filepath)

        if not transcribed_text:
            return jsonify({'error': 'Could not transcribe audio. Please ensure the audio file contains clear speech.'}), 400

        print(f"Transcribed: {transcribed_text}")

        # Process the query using Gemini AI
        response_text = process_query(transcribed_text)

        print(f"AI Response: {response_text}")

        # Convert text response to audio
        audio_buffer = text_to_speech(response_text)

        if not audio_buffer:
            return jsonify({'error': 'Failed to generate audio response'}), 500

        # Save audio to temporary file
        audio_filepath = os.path.join(app.config['UPLOAD_FOLDER'], 'response_audio.mp3')
        with open(audio_filepath, 'wb') as f:
            f.write(audio_buffer.read())

        # Return audio file
        return send_file(
            audio_filepath,
            mimetype='audio/mpeg',
            as_attachment=True,
            download_name='ai_response.mp3'
        ), 200

    except Exception as e:
        print(f"Error processing request: {e}")
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500
    finally:
        # Clean up uploaded file
        if uploaded_filepath and os.path.exists(uploaded_filepath):
            try:
                os.remove(uploaded_filepath)
            except:
                pass
        # Clean up generated audio file
        if audio_filepath and os.path.exists(audio_filepath):
            try:
                os.remove(audio_filepath)
            except:
                pass

@app.route('/process-text-to-audio', methods=['POST'])
def process_text_to_audio():
    """Endpoint to process text directly - returns audio response"""
    audio_filepath = None

    try:
        data = request.get_json()

        if not data or 'text' not in data:
            return jsonify({'error': 'No text provided'}), 400

        query = data['text']
        print(f"Received text query: {query}")

        # Process the query using Gemini AI
        response_text = process_query(query)

        print(f"AI Response: {response_text}")

        # Convert text response to audio
        audio_buffer = text_to_speech(response_text)

        if not audio_buffer:
            return jsonify({'error': 'Failed to generate audio response'}), 500

        # Save audio to temporary file
        audio_filepath = os.path.join(app.config['UPLOAD_FOLDER'], 'response_audio.mp3')
        with open(audio_filepath, 'wb') as f:
            f.write(audio_buffer.read())

        # Return audio file
        return send_file(
            audio_filepath,
            mimetype='audio/mpeg',
            as_attachment=True,
            download_name='ai_response.mp3'
        ), 200

    except Exception as e:
        print(f"Error processing request: {e}")
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500
    finally:
        # Clean up generated audio file
        if audio_filepath and os.path.exists(audio_filepath):
            try:
                os.remove(audio_filepath)
            except:
                pass

print("Flask app routes configured!")

Flask app routes configured!


In [ ]:
# Start Flask server with ngrok
def start_server(port=5002):
    """Start the Flask server"""
    print(f"Starting Flask server on port {port}...")
    app.run(host='0.0.0.0', port=port, debug=False, use_reloader=False)

# Start ngrok tunnel
def start_ngrok(port=5002):
    """Start ngrok tunnel"""
    try:
        # Set ngrok auth token directly
        ngrok.set_auth_token("<auth_token_here>")

        # Create tunnel
        public_url = ngrok.connect(port)
        print(f"\n{'='*60}")
        print(f"Ngrok tunnel created!")
        print(f"Public URL: {public_url}")
        print(f"{'='*60}\n")
        print("API Endpoints:")
        print(f"  Health Check: {public_url}/health")
        print(f"  Process Audio (returns text): {public_url}/process-audio (POST)")
        print(f"  Process Text (returns text): {public_url}/process-text (POST)")
        print(f"  Process Audio to Audio: {public_url}/process-audio-to-audio (POST)")
        print(f"  Process Text to Audio: {public_url}/process-text-to-audio (POST)")
        print(f"\nAI Model: Google Gemini")
        print(f"Response Format: JSON text (text endpoints) or MP3 audio (audio endpoints)")
        print(f"To stop the server, interrupt the kernel.\n")
        return public_url
    except Exception as e:
        print(f"Error starting ngrok: {e}")
        print("Make sure ngrok is installed: pip install pyngrok")
        return None

# Start server in background thread
SERVER_PORT = 5002
server_thread = threading.Thread(target=start_server, args=(SERVER_PORT,), daemon=True)
server_thread.start()

# Wait a moment for server to start
time.sleep(2)

# Start ngrok
ngrok_url = start_ngrok(SERVER_PORT)

Starting Flask server on port 5002...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5002
 * Running on http://172.28.0.12:5002
INFO:werkzeug:Press CTRL+C to quit



Ngrok tunnel created!
Public URL: NgrokTunnel: "https://unheededly-unangular-jill.ngrok-free.dev" -> "http://localhost:5002"

API Endpoints:
  Health Check: NgrokTunnel: "https://unheededly-unangular-jill.ngrok-free.dev" -> "http://localhost:5002"/health
  Process Audio (returns text): NgrokTunnel: "https://unheededly-unangular-jill.ngrok-free.dev" -> "http://localhost:5002"/process-audio (POST)
  Process Text (returns text): NgrokTunnel: "https://unheededly-unangular-jill.ngrok-free.dev" -> "http://localhost:5002"/process-text (POST)
  Process Audio to Audio: NgrokTunnel: "https://unheededly-unangular-jill.ngrok-free.dev" -> "http://localhost:5002"/process-audio-to-audio (POST)
  Process Text to Audio: NgrokTunnel: "https://unheededly-unangular-jill.ngrok-free.dev" -> "http://localhost:5002"/process-text-to-audio (POST)

AI Model: Google Gemini
Response Format: JSON text (text endpoints) or MP3 audio (audio endpoints)
To stop the server, interrupt the kernel.



## API Usage Examples

### Using curl to test the API:

```bash
# Process audio file (returns JSON text response)
curl -X POST <ngrok-url>/process-audio \
  -F "audio=@your_audio_file.wav"

# Process text (returns JSON text response)
curl -X POST <ngrok-url>/process-text -H "Content-Type: application/json" -d '{"text": "What is speech disorder?"}'

# Process audio file (returns MP3 audio response)
curl -X POST <ngrok-url>/process-audio-to-audio \
  -F "audio=@your_audio_file.wav" \
  --output response.mp3

# Process text (returns MP3 audio response)
curl -X POST <ngrok-url>/process-text-to-audio \
  -H "Content-Type: application/json" \
  -d '{"text": "What is speech disorder?"}' \
  --output response.mp3
```

### Using Python requests:

```python
import requests
import json

# Process audio file (get text response)
with open('your_audio.wav', 'rb') as f:
    response = requests.post(
        '<ngrok-url>/process-audio',
        files={'audio': f}
    )
    if response.status_code == 200:
        data = response.json()
        print(f"Transcribed: {data['transcribed_text']}")
        print(f"AI Response: {data['response']}")
    else:
        print(f"Error: {response.text}")

# Process text and get text response
response = requests.post(
    '<ngrok-url>/process-text',
    json={'text': 'What is speech disorder?'}
)
if response.status_code == 200:
    data = response.json()
    print(f"Query: {data['query']}")
    print(f"AI Response: {data['response']}")
else:
    print(f"Error: {response.text}")

# Process audio file (get audio response)
with open('your_audio.wav', 'rb') as f:
    response = requests.post(
        '<ngrok-url>/process-audio-to-audio',
        files={'audio': f}
    )
    if response.status_code == 200:
        with open('ai_response.mp3', 'wb') as out_file:
            out_file.write(response.content)
        print("Audio response saved to ai_response.mp3")
    else:
        print(f"Error: {response.text}")

# Process text and get audio response
response = requests.post(
    '<ngrok-url>/process-text-to-audio',
    json={'text': 'What is speech disorder?'}
)
if response.status_code == 200:
    with open('ai_response.mp3', 'wb') as f:
        f.write(response.content)
    print("Audio response saved to ai_response.mp3")
else:
    print(f"Error: {response.text}")
```

### Response Format:

**Text endpoints** (`/process-audio`, `/process-text`) return JSON:
```json
{
  "transcribed_text": "What is speech disorder?",  // Only for /process-audio
  "query": "What is speech disorder?",            // Only for /process-text
  "response": "A speech disorder is...",           // AI response from Gemini
  "ai_model": "Google Gemini"
}
```

**Audio endpoints** (`/process-audio-to-audio`, `/process-text-to-audio`) return MP3 audio file:
- Content-Type: `audio/mpeg`
- File name: `ai_response.mp3`
- Contains the AI's spoken response